In [ ]:
# Ex: 1

# In Terminal :
# pip install nltk spacy
# python -m spacy download en_core_web_sm


import nltk
from nltk.tokenize import word_tokenize, TweetTokenizer
from nltk.corpus import stopwords
import spacy

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("punkt_tab")

user_text = input("Enter a text to tokenize: ")

tokens = word_tokenize(user_text)
print("\nOriginal text:", user_text)
print("Word tokens:", tokens)

tweet_tokenizer = TweetTokenizer()
tweet_text = input("\nEnter a tweet text to tokenize: ")
tweet_tokens = tweet_tokenizer.tokenize(tweet_text)
print("\nOriginal tweet:", tweet_text)
print("Tweet tokens:", tweet_tokens)

stop_words = set(stopwords.words("english"))
clean_tokens = [token for token in tokens if token.lower() not in stop_words]
clean_text = " ".join(clean_tokens)
print("\nText after stopwords removal:", clean_text)

nlp = spacy.load("en_core_web_sm")
doc = nlp(user_text)
spacy_tokens = [token.text for token in doc]
print("\nspaCy tokens:", spacy_tokens)

result = [(token.lemma_, token.pos_) for token in doc]
print(result)

sentence_text = input("\nEnter a text for sentence segmentation: ")
doc = nlp(sentence_text)
sentences = [sentence.text for sentence in doc.sents]
print("\nSentences:", sentences)

In [ ]:
# Ex: 2

# pip install pandas pypdf2 matplotlib

import PyPDF2
import re
import string
import pandas as pd
import matplotlib.pyplot as plt

try:
    pdfFileObj = open("/content/resume-sample-pages.pdf", "rb")
except FileNotFoundError:
    print("File not found. Please check the file path and name.")
    exit()

pdfReader = PyPDF2.PdfReader(pdfFileObj)
num_pages = len(pdfReader.pages)

text = ""
for count in range(num_pages):
    pageObj = pdfReader.pages[count]
    text += pageObj.extract_text()

text = text.lower()
text = re.sub(r"\d+", "", text)
text = text.translate(str.maketrans("", "", string.punctuation))

terms = {
    "Quality/Six Sigma": [
        "black belt", "capability analysis", "control charts", "doe", "dmaic",
        "fishbone", "gage r&r", "green belt", "ishikawa", "iso", "kaizen", "kpi",
        "lean", "metrics", "pdsa", "performance improvement", "process improvement",
        "quality", "quality circles", "quality tools", "root cause", "six sigma",
        "stability analysis", "statistical analysis", "tqm"
    ],
    "Operations management": [
        "automation", "bottleneck", "constraints", "cycle time", "efficiency",
        "fmea", "machinery", "maintenance", "manufacture", "line balancing", "oee",
        "operations", "operations research", "optimization",
        "overall equipment effectiveness", "pfmea", "process", "process mapping",
        "production", "resources", "safety", "stoppage",
        "value stream mapping", "utilization"
    ],
    "Supply chain": [
        "abc analysis", "apics", "customer", "customs", "delivery", "distribution",
        "eoq", "epq", "fleet", "forecast", "inventory", "logistic", "materials",
        "outsourcing", "procurement", "reorder point", "rout", "safety stock",
        "scheduling", "shipping", "stock", "suppliers",
        "third party logistics", "transport", "transportation", "traffic",
        "supply chain", "vendor", "warehouse", "wip", "work in progress"
    ],
    "Project management": [
        "administration", "agile", "budget", "cost", "direction",
        "feasibility analysis", "finance", "kanban", "leader", "leadership",
        "management", "milestones", "planning", "pmi", "pmp", "problem",
        "project", "risk", "schedule", "scrum", "stakeholders"
    ],
    "Data analytics": [
        "analytics", "api", "aws", "big data", "business intelligence",
        "clustering", "code", "coding", "data", "database", "data mining",
        "data science", "deep learning", "hadoop", "hypothesis test", "iot",
        "internet", "machine learning", "modeling", "nosql", "nlp",
        "predictive", "programming", "python", "r", "sql", "tableau",
        "text mining", "visualization"
    ],
    "Healthcare": [
        "adverse events", "care", "clinic", "cphq", "ergonomics",
        "healthcare", "health care", "health", "hospital",
        "human factors", "medical", "near misses", "patient",
        "reporting system"
    ]
}

scores = {key: 0 for key in terms.keys()}

for area, keywords in terms.items():
    for word in keywords:
        if word in text:
            scores[area] += 1

summary = pd.DataFrame(scores.items(), columns=["Area", "Score"]).sort_values(
    by="Score", ascending=False
)

print(summary)

plt.figure(figsize=(10, 10))
plt.pie(
    summary["Score"],
    labels=summary["Area"],
    explode=(0.1, 0, 0, 0, 0, 0),
    autopct="%1.0f%%",
    shadow=True,
    startangle=90
)
plt.title("Industrial Engineering Candidate - Resume Decomposition by Areas")
plt.axis("equal")
plt.savefig("resume_screening_results.png")
plt.show()

In [ ]:
# Ex: 3

# pip install seaborn scikit-learn


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)

stop_words = set(stopwords.words("english"))
ps = PorterStemmer()

df = pd.read_csv("./datasets/IMDB-Dataset.csv")
df.dropna(inplace=True)

print("\nDataset Loaded Successfully")
print("Total Reviews:", len(df))
print("\nFirst 5 Records:")
print(df.head())

df["sentiment"] = df["sentiment"].str.capitalize()

def clean_text(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [ps.stem(w) for w in tokens if w.isalpha() and w not in stop_words]
    return " ".join(tokens)

df["cleaned_review"] = df["review"].apply(clean_text)

print("\nSample Cleaned Review:\n", df["cleaned_review"].iloc[0])

plt.figure(figsize=(6, 4))
sns.countplot(x="sentiment", data=df, palette="coolwarm")
plt.title("Distribution of Sentiments in Dataset")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    df["cleaned_review"],
    df["sentiment"],
    test_size=0.2,
    random_state=42
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("\nVocabulary size:", len(vectorizer.get_feature_names_out()))

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print("\nModel Evaluation Report:")
print(classification_report(y_test, y_pred))
print("Accuracy Score:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

cm = confusion_matrix(y_test, y_pred, labels=["Negative", "Positive"])
plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print("\nNumber of Positive reviews:", sum(y_test == "Positive"))
print("Number of Negative reviews:", sum(y_test == "Negative"))

user_review = input("\nEnter your review: ")
cleaned = clean_text(user_review)
user_vec = vectorizer.transform([cleaned])
prediction = model.predict(user_vec)[0]
print("\nPredicted Sentiment:", prediction)

feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

top_positive = np.argsort(coefficients)[-10:]
top_negative = np.argsort(coefficients)[:10]

print("\nTop words for Positive:")
print(feature_names[top_positive])

print("\nTop words for Negative:")
print(feature_names[top_negative])

In [ ]:
# Ex: 4

import math
import nltk
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from string import punctuation
import pandas as pd
import matplotlib.pyplot as plt

nltk.download("punkt")
nltk.download("stopwords")

print("\n=== KEYWORD EXTRACTION USING NLP (TF-IDF BASED) ===\n")

doc = input("Enter your text paragraph:\n")

stop_words = set(stopwords.words("english"))
sentences = sent_tokenize(doc)
words = word_tokenize(doc.lower())
filtered_words = [w for w in words if w not in stop_words and w not in punctuation]

print(f"Total Sentences : {len(sentences)}")
print(f"Total Words : {len(words)}")
print(f"Filtered Words : {len(filtered_words)}")

tf = {}
for word in filtered_words:
    tf[word] = tf.get(word, 0) + 1

for word in tf:
    tf[word] = tf[word] / len(filtered_words)

def count_sentences_containing(word, sentences):
    return sum(1 for sent in sentences if word in sent.lower())

idf = {}
for word in tf:
    idf[word] = math.log((1 + len(sentences)) / (1 + count_sentences_containing(word, sentences))) + 1

tf_idf = {word: tf[word] * idf[word] for word in tf}

sorted_keywords = sorted(tf_idf.items(), key=lambda x: x[1], reverse=True)[:15]

df_keywords = pd.DataFrame(sorted_keywords, columns=["Keyword", "TF-IDF Score"])

print("\nTop Keywords and their TF-IDF Scores:\n")
print(df_keywords.to_string(index=False))

word_freq = pd.Series(tf).sort_values(ascending=False)[:10]

plt.figure(figsize=(16, 12))

plt.subplot(2, 1, 1)
plt.barh(word_freq.index, word_freq.values, color='green')
plt.gca().invert_yaxis()
plt.title("Top 10 Frequent Words (Before TF-IDF)")
plt.xlabel("Frequency")
plt.ylabel("Words")

plt.subplot(2, 1, 2)
plt.barh(df_keywords["Keyword"], df_keywords["TF-IDF Score"])
plt.gca().invert_yaxis()
plt.title("Top Keywords by TF-IDF Score")
plt.xlabel("TF-IDF Score")
plt.ylabel("Keywords")
plt.show()

print("\nKeyword Extraction using NLP completed successfully.\n")

In [ ]:
# Ex: 5

german_to_english = {
    "hallo": "hello",
    "hi": "hi",
    "guten": "good",
    "morgen": "morning",
    "tag": "day",
    "abend": "evening",
    "nacht": "night",
    "willkommen": "welcome",
    "tschüss": "bye",
    "auf": "on",
    "wiedersehen": "goodbye"
}

def clean_text(sentence):
    cleaned = ""
    for ch in sentence:
        if ch.isalpha() or ch.isspace():
            cleaned += ch
        else:
            cleaned += " "
    return cleaned.strip().lower()

def tokenize(text):
    return [word for word in text.split() if word]

def translate_tokens(tokens):
    translated = []
    for word in tokens:
        if word in german_to_english:
            translated.append(german_to_english[word])
        else:
            translated.append(word)
    return translated

def reconstruct_sentence(translated_tokens):
    if not translated_tokens:
        return ""
    translated_tokens[0] = translated_tokens[0].capitalize()
    return " ".join(translated_tokens)

def translate_sentence(sentence):
    cleaned = clean_text(sentence)
    tokens = tokenize(cleaned)
    translated_tokens = translate_tokens(tokens)
    result = reconstruct_sentence(translated_tokens)
    return result


print("Simple German → English Translator (Built-in Python Only)")
print("Type 'exit' to quit.\n")

while True:
    german_sentence = input("Enter a German sentence: ").strip()
    if german_sentence.lower() == "exit":
        print("Goodbye!")
        break
    english_translation = translate_sentence(german_sentence)
    print("English translation:", english_translation, "\n")
    
""" 
# pip install googletrans==4.0.0rc1

from googletrans import Translator

translator = Translator()

source_text = input("Enter text to translate: ")
source_lang = input("Enter source language code (example: en): ")
dest_lang = input("Enter destination language code (example: fr): ")

translated = translator.translate(
    source_text,
    src=source_lang,
    dest=dest_lang
)

print("Translated text:", translated.text)
 """

In [ ]:
# Ex: 6

# pip install wordcloud vaderSentiment


import re
from collections import Counter
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

print("\n=== WHATSAPP CHAT ANALYSIS USING NLP ===\n")

file_path = "./datasets/WhatsApp Chat with SECAD2027B.txt"
with open(file_path, "r", encoding="utf-8") as f:
    chat = f.readlines()

dates = []
users = []
messages = []
hours = []

pattern = r"(\d+/\d+/\d+),\s(\d+):(\d+)\s?(am|pm)?\s-\s([^:]+):\s(.*)"

for line in chat:
    match = re.match(pattern, line, flags=re.IGNORECASE)
    if not match:
        continue

    date, hour, minute, ampm, user, msg = match.groups()

    hour = int(hour)
    if ampm:
        if ampm.lower() == "pm" and hour != 12:
            hour += 12
        elif ampm.lower() == "am" and hour == 12:
            hour = 0

    msg = re.sub(r"(http\S+|<Media omitted>|@\d+)", "", msg).strip()
    if not msg:
        continue

    users.append(user)
    messages.append(msg)
    hours.append(hour)
    dates.append(date)

if not messages:
    print("No valid messages found. Check file format (must be exported as .txt).")
else:
    print(f"\nTotal messages processed: {len(messages)}")
    print(f"Unique users detected: {len(set(users))}")

df = pd.DataFrame(
    {
        "Date": dates,
        "User": users,
        "Hour": hours,
        "Message": messages,
    }
)

top_user = df["User"].value_counts().idxmax()
print(f"Most active user: {top_user}")

avg_msgs = df.groupby("Date")["Message"].count().mean()
print(f"Average messages per day: {avg_msgs:.1f}")

all_text = " ".join(df["Message"]).lower()
words = re.findall(r"\b[a-zA-Z]+\b", all_text)
stopwords = {
    "the", "is", "and", "a", "to", "in", "of", "you", "i", "me", "it",
    "for", "on", "this", "that", "was", "we", "at"
}
clean_words = [w for w in words if w not in stopwords]

top_words = Counter(clean_words).most_common(10)
print("\nTop 10 Words:\n", top_words)

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color="white"
).generate(" ".join(clean_words))

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Word Cloud of Chat Messages")
plt.show()

analyzer = SentimentIntensityAnalyzer()
sentiments = {"Positive": 0, "Negative": 0, "Neutral": 0}

for msg in df["Message"]:
    score = analyzer.polarity_scores(msg)["compound"]
    if score > 0.05:
        sentiments["Positive"] += 1
    elif score < -0.05:
        sentiments["Negative"] += 1
    else:
        sentiments["Neutral"] += 1

print("\nSentiment Summary:", sentiments)

plt.figure(figsize=(5, 5))
plt.pie(
    sentiments.values(),
    labels=sentiments.keys(),
    autopct="%1.1f%%",
    startangle=90,
)
plt.title("Sentiment Distribution")
plt.show()

hour_count = Counter(df["Hour"])
plt.figure(figsize=(8, 4))
plt.bar(hour_count.keys(), hour_count.values())
plt.title("Messages by Hour of Day")
plt.xlabel("Hour (0–23)")
plt.ylabel("Message Count")
plt.grid(axis="y")
plt.show()

user_count = df["User"].value_counts().head(5)
plt.figure(figsize=(16, 4))
plt.barh(user_count.index, user_count.values)
plt.title("Top 5 Active Users")
plt.xlabel("Message Count")
plt.ylabel("User")
plt.gca().invert_yaxis()
plt.show()

print("\nWhatsApp Chat Analysis completed successfully.\n")

In [ ]:
# Ex: 7

import json
import random
import string
import datetime
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    words = nltk.word_tokenize(text)
    words = [lemmatizer.lemmatize(w) for w in words]
    return " ".join(words)

with open("./datasets/intents.json", "r", encoding="utf-8") as f:
    data = json.load(f)

patterns = []
tags = []
responses = {}

for intent in data["intents"]:
    tag = intent["tag"]
    responses[tag] = intent["responses"]
    for pattern in intent["patterns"]:
        patterns.append(clean_text(pattern))
        tags.append(tag)

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(patterns)

def get_response(user_input):
    cleaned_input = clean_text(user_input)
    user_vec = vectorizer.transform([cleaned_input])
    sim = cosine_similarity(user_vec, X)
    idx = sim.argmax()
    confidence = sim[0][idx]
    if confidence < 0.2:
        return "I'm not sure I understand. Could you please rephrase?", confidence
    tag = tags[idx]
    reply = random.choice(responses[tag])
    return reply, confidence

chat_history = []

def log_message(sender, message):
    chat_history.append(
        {
            "time": datetime.datetime.now().strftime("%H:%M:%S"),
            "sender": sender,
            "message": message,
        }
    )

print("ChatBuddy: Hello! I’m ChatBuddy, your NLP assistant.")
print("Type 'quit' to end the chat.\n")

message_count = 0

while True:
    user_input = input("You: ")
    if user_input.lower() in ["quit", "exit", "bye"]:
        print("\nChatBuddy: Goodbye! Have a great day!")
        break

    log_message("User", user_input)
    reply, confidence = get_response(user_input)
    log_message("ChatBuddy", reply)
    print(f"ChatBuddy: {reply}\n")
    message_count += 1

with open("./datasets/chat_log.txt", "w", encoding="utf-8") as f:
    for msg in chat_history:
        f.write(f"[{msg['time']}] {msg['sender']}: {msg['message']}\n")

print(f"\nTotal messages exchanged: {message_count}")
print("Chat history saved successfully to 'chat_log.txt'.")

In [ ]:
# Ex: 8

# pip install tensorflow


import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore  
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout  # type: ignore
from tensorflow.keras.models import Sequential, load_model # type: ignore
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.optimizers import Adam # type: ignore
import matplotlib.pyplot as plt
import pickle
import numpy as np
import re
import warnings

warnings.filterwarnings("ignore")

file_path = "./datasets/Sherlock Holmes.txt"
with open(file_path, "r", encoding="utf8") as f:
    text = f.read()

text = text.lower()
text = re.sub(r"[^a-zA-Z\s]", "", text)
text = re.sub(r"\s+", " ", text).strip()

print(f" Loaded text with {len(text.split())} words.")

tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

pickle.dump(tokenizer, open("token.pkl", "wb"))

total_words = len(tokenizer.word_index) + 1
print(f" Total unique words in dataset: {total_words}")

input_sequences = []
token_list = tokenizer.texts_to_sequences([text])[0]

for i in range(4, len(token_list)):
    seq = token_list[i - 4 : i + 1]
    input_sequences.append(seq)

max_seq_len = max(len(x) for x in input_sequences)
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_seq_len, padding="pre")
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = to_categorical(y, num_classes=total_words)

print(f" Total training samples: {X.shape[0]} | Sequence length: {max_seq_len}")

model = Sequential(
    [
        Embedding(total_words, 64, input_length=max_seq_len - 1),
        LSTM(256, return_sequences=True),
        Dropout(0.3),
        LSTM(256),
        Dense(256, activation="relu"),
        Dense(total_words, activation="softmax"),
    ]
)

model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"],
)

model.summary()

checkpoint = ModelCheckpoint(
    "best_next_word_model.h5", monitor="loss", save_best_only=True, verbose=1
)
early_stop = EarlyStopping(
    monitor="loss", patience=3, restore_best_weights=True
)

history = model.fit(
    X,
    y,
    epochs=10,
    batch_size=128,
    callbacks=[checkpoint, early_stop],
)

plt.plot(history.history["loss"], label="Loss")
plt.title("Training Loss Curve")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

model = load_model("best_next_word_model.h5")
print(" Model loaded successfully!")

def predict_next_word(model, tokenizer, seed_text):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences(
        [token_list], maxlen=max_seq_len - 1, padding="pre"
    )
    predicted = np.argmax(model.predict(token_list), axis=-1)[0]
    predicted_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted:
            predicted_word = word
            break
    return predicted_word

print("\n LSTM Text Generator Ready!")
print("Type a short phrase (at least 3 words). Type '1' to exit.\n")

while True:
    user_input = input("Enter your line: ").strip()
    if user_input == "1":
        print("\n Exiting program. Goodbye!")
        break
    if len(user_input.split()) < 3:
        print(" Please enter at least 3 words.\n")
        continue
    next_word = predict_next_word(model, tokenizer, user_input)
    print(f"\n Predicted next word: {next_word}\n")